# Label Revise

Use the 3D Slicer Jupyter extension in conjunction with Tkinter for simple interaction to perform batch plaque label revision.

Note: Please run each cell in the order of the code (using the “Run” button above).

1) Import the required modules:

In [1]:
import argparse as args
import os
import glob
import json
import sys
import math
import shutil
import time
import slicer
import tkinter as tk
from tkinter import *
import tkinter.messagebox as mbox
from tkinter.filedialog import askopenfilename, askdirectory
import cv2
from libtiff import TIFF 
import tifffile
import numpy as np
import pandas as pd
import random
import SimpleITK as sitk
import base64
from icon import img


ROOT_DIR = os.path.abspath("")
ori_dir = os.path.join(ROOT_DIR, "ori.mrk.json")

2) Functions included:

In [2]:
# Load the 'label.txt' file containing the labels to be modified. 
# You can modify the functions based on the specific meanings of each position in the label lines.
def load_txt(file_path, output_folfer):
    object_num = 0
    file_name = os.path.basename(file_path)
    for line in open(file_path, "r"):
        if line != [' ']:
            object_num = object_num+1
            line = line.strip()
            info = line.split(" ")
            x_c = float(info[0])
            y_c = float(info[1])
            z_c = float(info[2])
            w = float(info[3])
            h = float(info[4])
            l = float(info[5])
            cls = int(info[6])
            name = file_name.split(".")[0]
            folder = output_folfer + '/' + name + '/'
            if name not in os.listdir(output_folfer):
                os.mkdir(folder)
            copy_path = folder + 'json/'
            if "json" not in os.listdir(folder):
                os.mkdir(copy_path)
            copy_dir = os.path.join(copy_path, str(cls) + '-' + str(object_num) + '.mrk.json')
            shutil.copy(ori_dir, copy_dir)
            final_path = folder + 'json_new/'
            if "json_new" not in os.listdir(folder):
                os.mkdir(final_path)
            with open(copy_dir, "r", encoding='utf-8') as f:
                bbox = json.load(f)
            bbox['markups'][0]['center'] = [x_c,y_c,z_c]
            bbox['markups'][0]['controlPoints'][0]['position'] = [x_c, y_c, z_c]
            bbox['markups'][0]['size'] = [w, h, l]
            with open(copy_dir, "w", encoding='utf-8') as dump_f:
                json.dump(bbox, dump_f, indent=2)
    return file_name, folder, copy_path, final_path 


# Load the 'source.json' file as the original 'label.json' file, which can be overwritten.
def load_json(json_list):
    for json_dir in json_list:
        print(json_dir)
        slicer.util.loadMarkupsFiducialList(json_dir)    
        
        
# Export the '.json' label file in Slicer as a '.txt' file; you can modify the function to suit your required label format.
def json2txt(json_list, output_dir, img_w, img_h, img_l):
    df_3d_all = pd.DataFrame()
    i = 0
    for json_dir in json_list:
        json_name = os.path.basename(json_dir)
        cls = int(json_name[0])
        with open(json_dir, "r", encoding='utf-8') as f:
            bbox = json.load(f)
            center = bbox['markups'][0]['center']
            size = bbox['markups'][0]['size']
        x_c,y_c,z_c = center
        w,h,l = size
        x_r = w/2
        y_r = h/2
        z_r = l/2
        if (x_c<x_r)or((img_w-x_c)<x_r):
            if x_c<x_r:
                x_r = (x_r + x_c)/2
                x_c = x_r
            else:
                x_r = (img_w - x_c + x_r)/2
                x_c = img_w - x_r
            w = x_r*2
        if (y_c<y_r)or((img_h-y_c)<y_r):
            if y_c<y_r:
                y_r = (y_r + y_c)/2
                y_c = y_r
            else:
                y_r = (img_h - y_c + y_r)/2
                y_c = img_h - y_r
            h = y_r*2
        if (z_c<z_r)or((img_l-z_c)<z_r):
            if z_c<z_r:
                z_r = (z_r + z_c)/2
                z_c = z_r
            else:
                z_r = (img_l - z_c + z_r)/2
                z_c = img_l - z_r
            l =z_r*2
        df_3d_all[i] = pd.Series({"x_c": int(x_c),
                                  "y_c": int(y_c),
                                  "z_c": int(z_c),
                                  "w": int(w),
                                  "h": int(h),
                                  "l": int(l),
                                  "cls": cls })
        i = i+1
    df_3d_all.T.to_csv(output_dir, sep=' ',index=False, header=False)
    

# Read a TIF image.
def tiff_to_read(image_name): 
    tif = TIFF.open(image_name, mode = "r") 
    im_stack = list()
    for im in list(tif.iter_images()): 
        im_stack.append(im)
    return im_stack


# Use the '.txt' label file to annotate boxes in the TIF image.
def drawontif_c(file_name, folder, imgname, Thickness=1):
    file_suffix = os.path.splitext(file_name)[-1]
    if file_suffix != ".txt":
        mbox.askokcancel(title="warning", message="There is no output label file(.txt).\n Make sure you don't miss any steps above!")
        return 0
    txt_dir = os.path.join(folder, file_name)
    img = tiff_to_read(imgname)
    image = np.array(img)
    image_out = image
    for line in open(txt_dir, "r"):
        line = line.strip()
        info = line.split(" ")
        x_c = float(info[0])
        y_c = float(info[1])
        z_c = float(info[2])
        w = float(info[3])
        h = float(info[4])
        l = float(info[5])
        xmin = int(x_c-(w/2))
        xmax = int(x_c+(w/2))
        ymin = int(y_c-(h/2))
        ymax = int(y_c+(h/2))
        zmin = int(z_c-(l/2))
        zmax = int(z_c+(l/2))
        if xmin>xmax:
            xmin,xmax = xmax,xmin
        if ymin>ymax:
            ymin,ymax = ymax,ymin
        if zmin>zmax:
            zmin,zmax = zmax,zmin
        if xmin>=128:
            xmin = xmax = 127
        if ymin>=128:
            ymin = ymax = 127
        if zmin>=128:
            zmin = zmax = 127
        if xmax>=128:
            xmax = 127
        if ymax>=128:
            ymax = 127
        if zmax>=128:
            zmax = 127     
        for i in range(zmin, zmax+1):
            font = cv2.FONT_HERSHEY_SIMPLEX
            # The input parameters are: the image, the top-left coordinates, 
#             the bottom-right coordinates, the colour array and the line thickness.
            cv2.rectangle(image_out[i], (xmin,ymin), (xmax,ymax), (255,0,255), Thickness) 
            # The input parameters are an image, text, position, font, size, an array of colours, and line thickness.
            # cv2.putText(image_out[i], cls, (xmin,ymin), font, 0.3, (255,0,255), Thickness)            
    image_out = np.uint8(image_out)
    tifffile.imsave(os.path.join(folder, file_name.replace("txt", "tif").strip()), image_out)
    
    
# num+1
def addnum():
    num = int(Button.cget("text")) # Retrieve the component’s parameter options.
    Button.config(text=str(num + 1))
    
    
def printget():
    print("Input completed.")
    
    
# Get the current epoch.
def epochget():
    sc = scale_4.get()
    if sc >= 1:
        epoch.set(sc)
        print("Epoch completed: " + str(epoch.get()))

             
def validate_selection():
    current_epoch = scale_4.get()
    max_epoch = n.get()

    if max_epoch == 0:
        mbox.showwarning(title="warning", message="Please load label file first!")
    elif current_epoch < 1 or current_epoch > max_epoch:
        mbox.showwarning(title="warning", message=f"Epoch must be between 1 and {max_epoch}!")
    else:
        mbox.showinfo(title="remind", message=f"Selected epoch: {current_epoch}")
        
        
def loadimg():
    imgpath = askopenfilename(filetypes=[("IMAGES", "*.tif")])
    if imgpath != '':
        imgname.set(imgpath) 
    else:
        mbox.askokcancel(title="warning", message="Do not choose image!")
        print("do not choose file")
        

# Retrieve the size of the loaded image.
def imgsize():
    img = tiff_to_read(imgname.get())
    image = np.array(img)
    w, h, l = image.shape
    img_w.set(w) 
    img_h.set(h) 
    img_l.set(l) 
    

def loadfile():
    file_path = askopenfilename(filetypes=[("Text files", "*.txt")])
    if file_path != '':
        filepath.set(file_path) 
    else:
        mbox.askokcancel(title="warning", message="Do not choose file!")
        print("do not choose file")
        
        
def savedir():
    fileDir = askdirectory()
    if fileDir != '':
        dirpath.set(fileDir)
    else:
        mbox.askokcancel(title="warning", message="Do not choose save folder!")
        print("do not choose save folder")
        
        
def allinput(file_path, output_folfer):
    file_name, folder, copy_path, final_path  = load_txt(file_path, output_folfer) 
    # These are the text file name, the folder where the results are saved, and the folder containing the JSON and MRML files, respectively.
    
    json_list = glob.glob(os.path.join(copy_path, "*.mrk.json"))      # A list of all JSON file paths.
    
    output_dir = os.path.join(folder, file_name)                      # The path where the results are saved as a '.txt' file.
    return file_name, folder, copy_path, json_list, output_dir, final_path


def num(json_list):
    num_all = len(json_list)
    if num_all != 0:
        n = math.ceil(num_all / 30)
        rest = int(num_all % 30)
        return num_all, n, rest
    else:
        mbox.askokcancel(title="warning", message="The label file is empty,\n please choose a new one!")
        print("label file is empty, please choose new one")
        return 0, 0, 0
    

# Start a revision epoch; the value set here is 30. You can adjust the number of revisions per epoch, 
# but it is recommended to keep this below 50.
def epoch_start():   
    print(epoch.get())
    i = epoch.get() - 1
    slicer.util.mainWindow().moduleSelector().selectModule('Markups')
    slicer.util.loadVolume(imgname.get())
    mbox.askokcancel(title="remind", message="While you wanna have a break, remember to save\n slicer-supported-file(.mrb) with filename as 'eopch.mrml'(eg:1.mrb)")
    if i == (n.get()-1):        
        if rest.get() != 0:
            mbox.askokcancel(title="remind", message="It's the last epoch.")
            json_unit = json_list[(30*i) : (30*i+rest.get())]
            load_json(json_unit)
    else:
        mbox.askokcancel(title="remind", message="It's " + str(epoch.get()) + " epoch.")
        json_unit = json_list[(30*i) : (30*i+30)]
        load_json(json_unit)             
        

# Continue revising the current epoch.
def go_on(): 
    slicer.mrmlScene.Clear()
    mrml_list = glob.glob(os.path.join(final_path.get(), "*.mrb"))
    for mrml_dir in mrml_list:
        mrml_name = os.path.basename(mrml_dir)
        ename = mrml_name.split(".")[0]
        if int(ename) == epoch.get():
            slicer.util.loadScene(mrml_dir)    # load '.mrml' in last revise
            
            
# Automatically save the project file and the JSON files for all current labels.
def autosave():
    save_mrml = os.path.join(final_path.get(), str(epoch.get())+".mrb")
    slicer.util.saveScene(save_mrml)
    json_all = slicer.util.getNodesByClass("vtkMRMLMarkupsROINode")
    for i in json_all:
        slicer.util.saveNode(i, final_path.get()+i.GetName()+".mrk.json")
        
        
# Confirm and save.
def yes(): 
    autosave()
    mrml_list = glob.glob(os.path.join(final_path.get(), "*.mrb"))
    ename_list = list()
    for mrml_dir in mrml_list:
        t_save = time.localtime(os.stat(mrml_dir).st_mtime)
        t_now = time.localtime(time.time())
        mrml_name = os.path.basename(mrml_dir)
        ename = int(mrml_name.split(".")[0])
        ename_list.append(ename)
        if ename_list.count(epoch.get()):
            if (t_save.tm_year==t_now.tm_year)&(t_save.tm_mon==t_now.tm_mon)&(t_save.tm_mday==t_now.tm_mday)&(t_save.tm_hour<=t_now.tm_hour):
                print("saved completed.")
            else:
                mbox.askokcancel(title="remind", message="Make sure you already saved files in slicer!")
#         else:
#             mbox.askokcancel(title="warning", message="You didn't save files in slicer!!!")
            

# Exit the revise for the current epoch.
def exit():
    mbox.askokcancel(title="remind", message="Comfirm you wanna exit revise!")
    yes()
    json_list_1 = glob.glob(os.path.join(final_path.get(), "*.mrk.json"))
    json2txt(json_list_1, output_dir.get(), img_w.get(), img_h.get(), img_l.get())
    time.sleep(10)
    slicer.mrmlScene.Clear()
    
    
# Skip to the next epoch’s revision.
def nextepoch():
    yes()
    json_list_1 = glob.glob(os.path.join(final_path.get(), "*.mrk.json"))
    json2txt(json_list_1, output_dir.get(), img_w.get(), img_h.get(), img_l.get())
    time.sleep(50)
    slicer.mrmlScene.Clear()
    epoch_new = epoch.get()+1        #epoch+1
    epoch.set(epoch_new)
    time.sleep(50)
    epoch_start() 
    

# Labelling of patches identified during leak detection.
def addmiss():
    autosave()
    json_list_1 = glob.glob(os.path.join(final_path.get(), "*.mrk.json"))
    json2txt(json_list_1, output_dir.get(), img_w.get(), img_h.get(), img_l.get())
    time.sleep(50)
    slicer.mrmlScene.Clear()
    drawontif_c(file_name.get(), folder.get(), imgname.get())
    slicer.util.mainWindow().moduleSelector().selectModule('Markups')
    slicer.util.loadVolume(os.path.join(folder.get(), file_name.get().replace("txt", "tif").strip()))
    

# The final, complete save.
def saveall():
    json_list_1 = glob.glob(os.path.join(final_path.get(), "*.mrk.json"))
    json2txt(json_list_1, output_dir.get(), img_w.get(), img_h.get(), img_l.get())
    time.sleep(50)
    slicer.util.exit()
    image_out = drawontif_c(file_name.get(), folder.get(), imgname.get())
    
    
# Refresh the selection based on user input.
def refresh_epoch():
    print('Epoch refresh based on input.')
    # Operations that require data to be refreshed use the current time obtained from the time function.
    file_path = filepath.get()
    dir_path = dirpath.get()
    f_n, fd, c_p, j_l, o_d, f_p = allinput(file_path, dir_path)
    if (file_name != '') & (folder != '') & (copy_path != '') & (output_dir != ''):
        n_a, nn, r = num(j_l)
        global json_list
        file_name.set(f_n)
        folder.set(fd)
        copy_path.set(c_p)
        final_path.set(f_p)
        json_list = j_l
        output_dir.set(o_d)
        num_all.set(n_a)
        n.set(nn)
        rest.set(r)
            # A test.
#             nn = 3
#             n.set(3)
#             n_a = 2
#             num_all.set(2)
#             c_p = 'aabbcc'
#             copy_path.set('aabbcc')
            ###################
        lab_44_1 = Label(root, text="We set 30 annotations as a epoch, in this img, we got " + str(n_a) + " annotations,\n they're divided into " + str(nn) + " epochs, now choose which epoch to revise.", font=("Times", 15, "bold"))
        lab_44_1.place(x=300, y=290)
        scale_4.configure(to=nn)
        button4_1 = Button(root, text="✔", command=epochget)
        button4_1.place(x=835, y=360, width=30, height=20)
        lab_4_4_1 = Label(root, text="2) choose slicer folder as '" + c_p + "'.", font=("Times", 10, "bold") )  #copy_path交互
        lab_4_4_1.place(x=315, y=412, height=15)
        lab_6_3_1 = Label(root, text="'" + c_p + "'.", font=("Times", 10, "bold"))
        lab_6_3_1.place(x=280, y=603)

3) Creating an interactive interface using Tkinter:

In [3]:
json_list = list()

root = Tk()
root.title("Plaques Label Revise Tool")
root.geometry("1200x870")
root.resizable(width=False, height=False)
root.tk.call('tk', 'scaling', 1.5)  # Set DPI scaling.

# 创建主框架来容纳所有内容
main_frame = Frame(root)
main_frame.place(x=0, y=0, width=1200, height=870)

title = Label(root, text="Hello, let's start to revise!", font=("Times", 17, "bold"))
title.place(x=600, y=35, anchor='center')

global imgname, img_w, img_h, img_l, filepath, dirpath, file_name, folder, copy_path, output_dir, num_all, n, rest, epoch
# about img
imgname = tk.StringVar()
img_w = tk.IntVar()
img_h = tk.IntVar()
img_l = tk.IntVar()
# about files
filepath = tk.StringVar()
dirpath = tk.StringVar()
# about sets 1
file_name = tk.StringVar()
folder = tk.StringVar()
copy_path = tk.StringVar()
final_path = tk.StringVar()
output_dir = tk.StringVar()
num_all = tk.IntVar()
n = tk.IntVar()
rest = tk.IntVar()
# about sets 2
epoch = tk.IntVar() 

# input image road
lab_1 = Label(root, image="::tk::icons::information")
lab_1.place(x=270, y=60)
lab_11 = Label(root, text="Please input the road of image. (tif file)", font=("Times", 15, "bold"))
lab_11.place(x=300, y=60)
lab_1_1 = Label(root, text="img: ", font=("Times", 15, "bold"))
lab_1_1.place(x=300, y=95)
ent_1_1 = Entry(root, textvariable = imgname)
ent_1_1.place(x=350, y=100, width=480, height=20)
button_1_1 = Button(root, text="load", command=lambda:[loadimg(),imgsize()])
button_1_1.place(x=835, y=100, width=35, height=20)

# choose label .txt file
lab_2 = Label(root, image="::tk::icons::information")
lab_2.place(x=270, y=135)
lab_22 = Label(root, text="Please input the road of label of image. (txt file)", font=("Times", 15, "bold"))
lab_22.place(x=300, y=135)
lab_2_1 = Label(root, text="file: ", font=("Times", 15, "bold"))
lab_2_1.place(x=300, y=170)
ent_2_1 = Entry(root, textvariable = filepath)
ent_2_1.place(x=350, y=175, width=480, height=20)
button2_1 = Button(root, text="load", command=loadfile)
button2_1.place(x=835, y=175, width=35, height=20)
# choose save path
lab_3 = Label(root, image="::tk::icons::information")
lab_3.place(x=270, y=210)
lab_33 = Label(root, text="Please input folder road to save new label. (save folder)", font=("Times", 15, "bold"))
lab_33.place(x=300, y=210)
lab_3_1 = Label(root, text="folder: ", font=("Times", 15, "bold"))
lab_3_1.place(x=300, y=250)
ent_3_1 = Entry(root, textvariable = dirpath)
ent_3_1.place(x=370, y=255, width=460, height=20)
# button3_1 = Button(root, text="load", command=savedir)
button3_1 = Button(root, text="load", command=lambda:[savedir(),refresh_epoch()])
button3_1.place(x=835, y=255, width=35, height=20)

# root.after(100, refresh_epoch)

# revise input
lab_4 = Label(root, image="::tk::icons::information")
lab_4.place(x=270, y=290)
lab_44 = Label(root, text="We set 50 annotations as a epoch, in this img, we got ? annotations,\n they're divided into ? epochs, now choose which epoch to revise.", font=("Times", 15, "bold"))
lab_44.place(x=300, y=290)
lab_4_1 = Label(root, text="epoch: ", font=("Times", 15, "bold"))
lab_4_1.place(x=300, y=355)
scale_4 = tk.Scale(root, from_=1, to=10, length=465, width=20, orient=tk.HORIZONTAL)  
scale_4.place(x=365, y=337)

# Add validation button
# button_validate = Button(root, text="validation", command=validate_selection)
# button_validate.place(x=870, y=360, width=40, height=20)

lab_4_2 = Label(root, image="::tk::icons::warning")
lab_4_2.place(x=277, y=393)
lab_4_3 = Label(root, text="Before revising, please note that: 1) after revising, you must confirm you saved all revises in slicer.", font=("Times", 10, "bold"))
lab_4_3.place(x=315, y=396, height=15)
lab_4_4 = Label(root, text="2) choose slicer save folder as '...'.", font=("Times", 10, "bold") )  
lab_4_4.place(x=315, y=412, height=15)

# root.after(100, refresh_start)

# start revise
lab_5 = Label(root, image="::tk::icons::information")
lab_5.place(x=240, y=450)
lab_5_1 = Label(root, text="When you revise this epoch for the first time, please select 'Confirm Start',\n else you continue revise this epoch from the last break, please select 'Continue'.", font=("Times", 15, "bold"))
lab_5_1.place(x=268, y=450)
button5_1 = Button(root, text=" Confirm Start ", font=("Times", 14, "bold"), command=epoch_start)   #command=epoch_start
button5_1.place(x=470, y=525, width=180, height=30, anchor='center')
button5_2 = Button(root, text=" Continue ", font=("Times", 14, "bold"), command=go_on)        #command=go_on
button5_2.place(x=770, y=525, width=180, height=30, anchor='center')

lab_6 = Label(root, text="--------------------------------------------------------------------------------------------------------------------", font=("Times", 15, "bold"))
lab_6.place(x=600, y=560, anchor='center')
lab_6_1 = Label(root, image="::tk::icons::question")
lab_6_1.place(x=240, y=585)
lab_6_2 = Label(root, text="Make sure you have saved markups(.mrk.json)&slicer-supported-file(.mrb) in slicer at road: ", font=("Times", 11, "bold"))
lab_6_2.place(x=280, y=583)
lab_6_3 = Label(root, text="'...'.", font=("Times", 10, "bold"))
lab_6_3.place(x=280, y=603)
button6_1 = Button(root, text=" Yes ", font=("Times", 14, "bold"), command=yes)   
button6_1.place(x=915, y=588, width=50, height=30)

# stop epoch
lab_7 = Label(root, image="::tk::icons::information")
lab_7.place(x=270, y=645)
lab_77 = Label(root, text="Exit revise or continue next epoch.", font=("Times", 15, "bold"))
lab_77.place(x=305, y=645)
button7_1 = Button(root, text=" Exit ", font=("Times", 14, "bold"), command=exit)   #command=exit
button7_1.place(x=670, y=660, width=70, height=30, anchor='center')
lab_7_1 = Label(root, text="or", font=("Times", 15, "bold"))
lab_7_1.place(x=723, y=645)
button7_2 = Button(root, text=" Next ", font=("Times", 14, "bold"), command=nextepoch)   #command=nextepoch
button7_2.place(x=800, y=660, width=70, height=30, anchor='center')

# Determine the missed plaques
lab_8 = Label(root, image="::tk::icons::information")
lab_8.place(x=240, y=695)
lab_88 = Label(root, text="After you finish all epochs, comfirm there is no missing items in this image.", font=("Times", 15, "bold"))
lab_88.place(x=270, y=695)
button8_1 = Button(root, text=" Add ", font=("Times", 14, "bold"), command=addmiss)    #command=addmiss
button8_1.place(x=915, y=695, width=50, height=30)

lab_9 = Label(root, text="--------------------------------------------------------------------------------------------------------------------", font=("Times", 15, "bold"))
lab_9.place(x=600, y=742, anchor='center')
lab_9_1 = Label(root, image="::tk::icons::question")
lab_9_1.place(x=240, y=762)
lab_9_2 = Label(root, text="Make sure again you have saved all markups(.mrk.json)&slicer-supported-file(.mrb) in slicer. ", font=("Times", 11, "bold"))
lab_9_2.place(x=280, y=767)
button9_1 = Button(root, text=" Yes ", font=("Times", 14, "bold"), command=yes)     
button9_1.place(x=915, y=765, width=50, height=30)
lab_9_3 = Label(root, image="::tk::icons::information")
lab_9_3.place(x=430, y=805)
lab_9_2 = Label(root, text="Finish and Exit revise.", font=("Times", 15, "bold"))
lab_9_2.place(x=465, y=805)
button9_1 = Button(root, text=" ✔ ", font=("Times", 14, "bold"), command=saveall)   #command=saveall
button9_1.place(x=690, y=820, width=40, height=30, anchor='center')


tmp = open('tmp.ico', 'wb+')
tmp.write(base64.b64decode(img))
tmp.close()
root.iconbitmap('tmp.ico')
os.remove('tmp.ico')


mainloop()
# root.lift()
# root.focus_force()
# root.update()
